# 进阶实践 2：头文件与多文件编程

> **运行位置**：短实验在在线 C17 内核中运行；下载的完整程序与编译命令在本地终端运行。在线 `scanf`、`getchar`、`fgets(..., stdin)` 仍不能交互读取键盘。文件流与标准输入不是同一个对象。

先学完前六章，再完成本专题。先预测、运行、修改，最后解释；修改函数或类型定义后重启内核并从头运行。

**学习目标**：区分预处理、编译和链接；把接口放进头文件；用 enum 表达错误状态；用 typedef 简化类型名字；通过独立调用程序验证模块。

完整示例由多个真实源文件组成。在线只实验类型与常量，文件的编译和链接在本地完成。

## 1. 为什么拆成多个文件？

第6章已经把“统计”与“显示”拆成函数。本专题再把它们放进不同文件，让同一份统计实现既能供报告程序调用，也能供验收程序调用。

```text
grades.h  声明类型、常量与接口
   ↑                 ↑
main.c            grades.c
显示报告          实现统计与内部辅助函数
   │                 │
 main.o           grades.o
       \           /
          链接
       grade_report
```

头文件通过 `#include` 参与预处理；每个 .c 文件连同包含进来的内容分别编译，形成各自的翻译单元。链接器把目标文件中的引用与定义连接起来。头文件本身不需要单独当作源文件编译。

## 2. 从“有效/无效”扩展为明确状态

enum 给一组整数常量命名，避免让调用者猜测返回数字是什么意思。typedef 为已有类型引入别名，并不自动创建新的、不兼容的类型。

下面只演示声明和使用。预测输出，再把状态改为 DEMO_EMPTY。完整项目会区分空数据、非法成绩和人数过多。

In [ ]:
#include <stdio.h>
{
    typedef enum { DEMO_OK, DEMO_EMPTY } DemoStatus;
    typedef struct { DemoStatus status; double average; } DemoReport;
    DemoReport report = {DEMO_OK, 77.67};
    if (report.status == DEMO_OK) { printf("平均分=%.2f\n", report.average); }
    else { printf("没有数据\n"); }
}

## 3. 宏与有类型的对象

`#define` 在预处理阶段进行替换，不声明变量，也没有 C 变量的块作用域。`const int` 则声明有类型的只读对象；二者不可以在所有语境中互换。

项目的 GRADE_CAPACITY 使用简单的整数宏，所有调用者共享人数上限。初学阶段不使用带副作用的函数式宏；需要计算时优先写普通函数。

下面同样的常量出现两次，是为了观察替换与对象声明的区别。预测后运行，再解释修改宏值为什么需要重新编译完整程序。

In [ ]:
#define COURSE_DEMO_LIMIT 100
{
    const int local_limit = COURSE_DEMO_LIMIT;
    printf("宏值=%d 对象值=%d\n", COURSE_DEMO_LIMIT, local_limit);
}
#undef COURSE_DEMO_LIMIT

## 4. 下载完整模块

把以下四个文件下载到同一目录，保留文件名：

| 文件 | 阅读重点 |
| --- | --- |
| [grades.h](practice/modules/grades.h) | 公共类型、人数上限、函数声明、输入和错误约定 |
| [grades.c](practice/modules/grades.c) | 统计函数定义、内部辅助函数 |
| [main.c](practice/modules/main.c) | 调用接口并显示结果 |
| [grade_checks.c](practice/modules/grade_checks.c) | 独立验收入口，覆盖8组输入 |

先读 grades.h 的接口注释，再读 main.c，最后进入 grades.c。函数定义所在的 .c 也包含自己的头文件，让编译器核对声明与定义是否一致。

`static int score_is_valid(...)` 使这个函数具有内部链接，只供当前翻译单元使用。不要把它与“函数内 static 局部变量具有静态存储期”混为一谈；本模块不依靠持久的可变状态工作。

## 5. 头文件保护与声明、定义

grades.h 用 `#ifndef / #define / #endif` 防止在同一个翻译单元里重复引入内容。验收程序故意包含它两次，应仍能编译。

头文件放共享类型和函数声明；本例的普通函数定义只在 grades.c 中出现一次。不要写 `#include "grades.c"` 代替链接，也不要把同一个 main 和另一个 main 一起链接。

头文件保护不能解决“多个 .c 中重复定义同一个外部函数”的问题：预处理分别发生在各翻译单元内。修改头文件后，需要重新编译所有依赖它的 .c 文件。

## 6. 实践：分别编译，再链接

在下载目录运行：

```bash
cc -std=c17 -Wall -Wextra -Wpedantic -Werror -c main.c -o main.o
cc -std=c17 -Wall -Wextra -Wpedantic -Werror -c grades.c -o grades.o
cc main.o grades.o -o grade_report
./grade_report
```

Windows 运行 grade_report.exe。预期输出：

```text
平均分=77.67 最高分=90 及格=2
```

也可以一次驱动编译和链接：

```bash
cc -std=c17 -Wall -Wextra -Wpedantic -Werror main.c grades.c -o grade_report
```

`.o` 是目标文件，不是可直接运行的完整程序。命令行需要列出参与链接的 .c 或 .o；仅包含头文件并不会自动把对应实现加入链接。

## 7. 运行模块验收

让另一个入口调用同一个 grades.c：

```bash
cc -std=c17 -Wall -Wextra -Wpedantic -Werror -c grade_checks.c -o grade_checks.o
cc grade_checks.o grades.o -o grade_checks
./grade_checks
```

Windows 运行 grade_checks.exe。应输出8行 PASS，最后为 `8/8 PASS`，正常退出。

| 用例 | 应有行为 |
| --- | --- |
| 85、90、58 | 平均77.67、最高90、及格2 |
| 单人成绩60 | 平均60、最高60、及格1 |
| 0、100 | 平均50、最高100、及格1 |
| 空数据 | GRADE_EMPTY |
| 85、-1 | GRADE_INVALID，不返回部分报告 |
| 101 | GRADE_INVALID |
| 空指针且人数为1 | GRADE_INVALID，不访问空指针 |
| 人数101 | GRADE_TOO_MANY，在访问数组前拒绝 |

所有错误状态的数值字段都为0。先看到参考实现全部通过，再自行修改实现或扩展接口；每次修改 grades.c 后重新编译 grades.o。

## 8. 分层练习

### A. 识别错误阶段

在副本里依次尝试以下改动，每次完成后恢复。记录是编译失败、链接失败，还是运行结果不符合预期。

1. 在 main.c 中把函数名字拼错。
2. 链接时只写 `cc main.o -o grade_report`。
3. 把 grades.c 的及格条件改成大于60。

<details><summary>提示</summary>

依次观察调用处是否有声明、链接时是否有定义、边界60是否仍及格。

</details>

<details><summary>参考答案</summary>

严格编译下1应失败；2缺少统计函数定义而链接失败；3可能正常编译链接，但验收中的单人成绩60会 FAIL。

</details>

### B. 先写验收，再增加最低分

先在 grade_checks.c 设计正常、单人、0/100和错误输入的期望，再给 GradeReport 加最低分字段并更新 grades.c。

验收：默认最低58、单人60的最低60、0/100的最低0；错误结果的数值字段仍全为0。更新所有结构体初始化器与结果比较，不能只新增字段而不检查它。

<details><summary>一级提示</summary>

最低分不能像最高分一样总从0开始，否则全是正分时会得到不存在的0分。

</details>

<details><summary>二级提示</summary>

确认至少有一个可读取元素后，用第一个成绩初始化；仍需校验每个成绩。只在全部成功时将统计量填入最终结果。

</details>

### C. 重新实现模块

保留 grades.h、main.c、grade_checks.c 的接口不变，在自己的 grades.c 中重写统计函数。以8/8 PASS为起点，再补充59/60、重复最高分、100人成绩的验收。

### D. 跨专题挑战

让文件报告程序调用 grades_summarize，读取部分负责收集数组，统计部分复用模块，输出部分保存报告。保留全部错误检查；不要同时保留两套统计算法。

验收：默认报告数值不变；空文件、非法成绩、超过100人仍明确失败；编译时需要链接 grades.c。完成后画出“输入 → 统计 → 输出”的函数调用关系。

In [ ]:
// 作答记录：任选一个错误，记录阶段、原因和修复后的验收结果。

## 小结

头文件描述调用约定，源文件提供实现，链接把各部分组合起来。用返回结果传递状态，让统计模块只读取输入；同一份实现可以被多个入口独立验证。